# 5.2. Stage 5: Monthly Aggregation — Unique Vacancies

**Stage:** 5.2 of 5

**Purpose:** Aggregate the daily unique pickles produced by Stage 5.1 into monthly Parquet and JSON files. For each calendar month, all daily files belonging to that month are concatenated into a single dataset. A Ukrainian-only export is also saved for each month.

**Input:** `data/stage_05/intermediate/pkl_daily_unique/ua-YYYY-MM-DD.pkl`

**Output:**
- `data/stage_05/processed/parquet_monthly_unique/YYYY-M.parquet` — monthly Parquet (all languages)
- `data/stage_05/processed/json_monthly_unique/YYYY-M.json` — monthly JSON (all languages)
- `data/stage_05/processed/json_monthly_ua_unique/YYYY-M.json` — monthly JSON (Ukrainian only)

**Run:** Set `start_date` and `end_date` to cover the months you want to generate, then execute all cells top to bottom.

In [4]:
import datetime
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

Enable autoreload, add `code/` to the Python path, and trigger memory cleanup.

In [2]:
import stage1 as st1
import stage2 as st2
import stage3 as st3
import pandas as pd

G:\mendely-paper-repository\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import stage modules and pandas. `stage1`, `stage2`, `stage3` are imported so their Config paths are available if needed during inspection.

In [3]:
import pandas as pd
process_df = pd.read_pickle(g.Config.STAGE5_PROCESS_UNIQUE_PATH)
process_df = process_df.sort_values(by='input_path').reset_index(drop=True)
process_df

,input_path,region_path,rejoin_path,rejoin_status
0,ua-2024-01-01,../data/stage_01/intermediate/id_region_click\...,../data/stage_05/intermediate/pkl_daily_unique...,complete


Load the Stage 5 unique process tracker and sort by filename to ensure files are processed in chronological order.

In [4]:
import pandas as pd

start_date = "2024-01-01"
end_date = "2024-01-31"

date_range = pd.date_range(start=start_date, end=end_date, freq='ME')
df = pd.DataFrame({'year': date_range.year, 'month': date_range.month})
df.head(3)

,year,month
0,2024,1


Define the date range to process. `date_range` generates one entry per calendar month end between `start_date` and `end_date`. Adjust these dates to cover the full dataset before running the main loop.

> **Demo:** The default range `2024-01-01` to `2024-01-31` covers the single demo file.

In [5]:
pd.read_pickle(process_df.loc[0, "rejoin_path"])

,id,number_of_clicks,date,region_original,city,district,region,country,latitude,longitude,...,job_type_score,classified_code,classified_title,skill_labels_en,classified_title_clean,extract_type,esco_title,esco_skills,esco_id,esco_code
0,2673941879650338041,210.0,2024-01-01,Sumy,Sumy,,Sumy Oblast,Ukraine,50.9077,34.7981,...,0.4700,2411,Accountants,"['carry out job analysis', 'motivate employees...",accountants,preferredLabel_fuzzy,accountant,"[""public finance"", ""financial forecasting"", ""b...",http://data.europa.eu/esco/occupation/78db356f...,2411.1.11
1,2829269335131763744,412.0,2024-01-01,Mykolaiv,Mykolaiv,,Mykolaiv Oblast,Ukraine,46.975,31.994,...,0.4894,2411,Accountants,"['consult with technical staff', 'communicate ...",accountants,preferredLabel_fuzzy,accountant,"[""public finance"", ""financial forecasting"", ""b...",http://data.europa.eu/esco/occupation/78db356f...,2411.1.11
2,2292953494970979965,428.0,2024-01-01,Chernihiv,Chernihiv,,Chernihiv Oblast,Ukraine,51.4982,31.2893,...,0.4967,2411,Accountants,['provide training on quality management super...,accountants,preferredLabel_fuzzy,accountant,"[""public finance"", ""financial forecasting"", ""b...",http://data.europa.eu/esco/occupation/78db356f...,2411.1.11
3,8991901255055716088,107.0,2024-01-01,Rivne,Rivne,,Rivne Oblast,Ukraine,50.6199,26.2516,...,0.4720,2132,Software developers,"['cooperate with colleagues', 'manage sales te...",software developers,altLabels,software developer,"[""embedded systems"", ""computer programming"", ""...",http://data.europa.eu/esco/occupation/57af9090...,2514.2.1
4,9937412981206494120,84.0,2024-01-01,Lviv,Lviv,,Lviv Oblast,Ukraine,49.8397,24.0297,...,0.4277,2421,Business analysts,"['innovation processes', 'create software desi...",business analysts,preferredLabel_fuzzy,business analyst,"[""digital data processing"", ""business analysis...",http://data.europa.eu/esco/occupation/60082a99...,2421.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,5682536959350677812,191.0,2024-01-01,Odesa,Odesa,,Odesa Oblast,Ukraine,46.4825,30.7233,...,0.4900,1324,Warehouse manager,['interact professionally in research and prof...,warehouse manager,preferredLabel,warehouse manager,"[""warehouse operations"", ""safety regulations f...",http://data.europa.eu/esco/occupation/2f5de1ab...,1324.3.4
96,4745382362973408538,226.0,2024-01-01,Ternopil,Ternopil,,Ternopil Oblast,Ukraine,49.5535,25.5948,...,0.4246,1330,Warehouse manager,"['cooperate with colleagues', 'manage sales te...",warehouse manager,preferredLabel,warehouse manager,"[""warehouse operations"", ""safety regulations f...",http://data.europa.eu/esco/occupation/2f5de1ab...,1324.3.4
97,1748822233724604818,16.0,2024-01-01,Zhytomyr,Zhytomyr,,Zhytomyr Oblast,Ukraine,50.2547,28.6587,...,0.4480,1324,Warehouse manager,['provide training on quality management super...,warehouse manager,preferredLabel,warehouse manager,"[""warehouse operations"", ""safety regulations f...",http://data.europa.eu/esco/occupation/2f5de1ab...,1324.3.4
98,5942472651725435914,426.0,2024-01-01,Dnipro,Dnipro,,Dnipropetrovsk Oblast,Ukraine,48.4647,35.0462,...,0.4383,1324,Warehouse manager,"['consult with technical staff', 'communicate ...",warehouse manager,preferredLabel,warehouse manager,"[""warehouse operations"", ""safety regulations f...",http://data.europa.eu/esco/occupation/2f5de1ab...,1324.3.4


**Inspection cell** — loads the first rejoin pickle to verify the column schema before running the aggregation loop.

In [6]:
g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_JSON_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_PARQUET_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_JSON_UA_PATH)

Ensure all three monthly output folders exist before the aggregation loop writes to them.

In [7]:
import os

selected_columns = ["id", "date", "clean_title", "clean_desc", "min_salary", "max_salary", "currency", "salary_rate", "date_created", "date_expired", "desc_lang",'skill_ids', "skill_labels", "skill_labels_en","job_type", "classified_code", "classified_title", 'classified_title_clean', 'extract_type', 'esco_title', 'esco_skills', 'esco_id', 'esco_code', "number_of_clicks", "region_original", "region", "city", "country", "latitude", "longitude"]

for _, row in df.iterrows():

    month_data = pd.DataFrame(columns=selected_columns)
    print(f"Month: {df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.")

    started = False
    for __, prow in process_df.iterrows():

        if pd.isna(prow["rejoin_path"]):
            print(f"Skipping row with missing path: {prow.name}")
            continue

        # Verify path exists
        if not os.path.exists(prow["rejoin_path"]):
            print(f"File not found: {prow['rejoin_path']}")
            continue


        data = pd.read_pickle(prow["rejoin_path"])
        data["date"] = pd.to_datetime(data["date"])
        y = data.loc[0, "date"].year
        m = data.loc[0, "date"].month

        if y == row["year"] and m == row["month"]:
            print(f"{data.loc[0, "date"]}: working...")
            if month_data.shape[0] == 0:
                started = True
                
# Make column selection robust against schema changes

                existing_columns = [col for col in selected_columns if col in data.columns]
                missing_columns = [col for col in selected_columns if col not in data.columns]

                if missing_columns:
                    print(f"Warning: missing columns in data: {missing_columns}")

# Create a safe copy with only existing columns
                month_data = data.reindex(columns=selected_columns).copy()
                continue
            else:
                data = data[selected_columns]
                
# Make column selection robust against schema changes

                existing_columns = [col for col in selected_columns if col in data.columns]
                missing_columns = [col for col in selected_columns if col not in data.columns]

                if missing_columns:
                    print(f"Warning: missing columns in data: {missing_columns}")

# Create a safe copy with only existing columns
                data = data.reindex(columns=selected_columns).copy()
                month_data = pd.concat([month_data, data], ignore_index=True)
        else:
            print(f"{data.loc[0, "date"]}: stop...")
            if started:
                break

    if month_data.shape[0] > 0:
        json_path = os.path.join(g.Config.STAGE5_MONTHLY_UNIQUE_JSON_PATH, f"{df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.json")
        month_data.to_json(json_path, orient="records")

        parquet_path = os.path.join(g.Config.STAGE5_MONTHLY_UNIQUE_PARQUET_PATH, f"{df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.parquet")
        month_data.to_parquet(parquet_path)

        month_data = month_data[month_data["country"] == "Ukraine"]
        ua_json_path = os.path.join(g.Config.STAGE5_MONTHLY_UNIQUE_JSON_UA_PATH, f"{df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.json")
        month_data.to_json(ua_json_path, orient="records")

        print(f"- date {df.loc[_ ,"year"]}-{df.loc[_ ,"month"]} processing complete!")
print("All done!")

Month: 2024-1.
2024-01-01 00:00:00: working...
- date 2024-1 processing complete!
All done!


**Main monthly aggregation loop** — for each calendar month in `df`, iterates through all daily rejoin pickles in the tracker and concatenates those matching that month into a single `month_data` DataFrame.

`selected_columns` defines the output schema — only these columns are kept in the monthly files. Missing columns are handled gracefully with a warning rather than an error.

Outputs per month:
- `.parquet` — columnar format for efficient downstream analysis
- `.json` — full dataset (all languages)
- `_ua.json` — Ukrainian-language vacancies only (`country == "Ukraine"`)

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.